# Notebook 02 — Demand Prediction Agent

Predicts:
- **Charging demand** (occupied chargers)
- **Station utilization** (occupancy / capacity)
- **Congestion probability** (utilization ≥ 80%)
- **Expected charging load** (kWh)

**Evaluation metrics (OP'26):** RMSE, MAE, R²

In [ ]:
import sys
from pathlib import Path
import json
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

import config
from helpers.preprocessing import time_based_split, handle_missing_values
from helpers.features import (
    add_time_features, add_lag_features, add_station_features,
    get_feature_matrix,
    TARGET_UTILIZATION, TARGET_DEMAND, TARGET_LOAD, TARGET_CONGESTION,
)
from helpers.metrics import regression_metrics, congestion_metrics

print("Loading processed UrbanEV data...")

In [ ]:
# Load preprocessed data from Notebook 01
df = pd.read_parquet(config.DATA_PROCESSED / "urbanev_hourly.parquet")
df = handle_missing_values(df)
print("Rows:", len(df), "| Stations:", df["station_id"].nunique())

## 1. Feature Engineering

In [ ]:
df = add_time_features(df)
df = add_station_features(df)
df = add_lag_features(df, lags=[1, 24])

# Drop rows where lags are missing (start of each station's series)
df = df.dropna().reset_index(drop=True)
print("Rows after lag features:", len(df))
df.head()

In [ ]:
# Save feature table for downstream notebooks
df.to_parquet(config.DATA_FEATURES / "training_features.parquet", index=False)

## 2. Time-Based Train / Validation / Test Split
Never shuffle time-series — future data must not leak into training.

In [ ]:
train, val, test = time_based_split(df, config.TRAIN_RATIO, config.VAL_RATIO)
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Train dates: {train['datetime'].min()} → {train['datetime'].max()}")
print(f"Test dates:  {test['datetime'].min()} → {test['datetime'].max()}")

In [ ]:
X_train, feature_cols = get_feature_matrix(train)
X_val, _ = get_feature_matrix(val)
X_test, _ = get_feature_matrix(test)

print("Features used:", feature_cols)

## 3. Train Models (sklearn Random Forest — interpretable, no deep learning)

In [ ]:
# --- Utilization model (regression) ---
model_util = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_util.fit(X_train, train[TARGET_UTILIZATION])

# --- Demand model (regression) ---
model_demand = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_demand.fit(X_train, train[TARGET_DEMAND])

# --- Load / kWh model (regression) ---
model_load = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_load.fit(X_train, train[TARGET_LOAD])

# --- Congestion model (classification) ---
model_congestion = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model_congestion.fit(X_train, train[TARGET_CONGESTION])

print("All 4 models trained.")

## 4. Evaluate on Test Set (OP'26 Metrics)

In [ ]:
pred_util = model_util.predict(X_test)
pred_demand = model_demand.predict(X_test)
pred_load = model_load.predict(X_test)
pred_congestion_prob = model_congestion.predict_proba(X_test)[:, 1]

metrics = {
    "utilization": regression_metrics(test[TARGET_UTILIZATION], pred_util),
    "demand": regression_metrics(test[TARGET_DEMAND], pred_demand),
    "load_kwh": regression_metrics(test[TARGET_LOAD], pred_load),
    "congestion": congestion_metrics(test[TARGET_CONGESTION], pred_congestion_prob),
}

metrics_df = pd.DataFrame(metrics).T
print("=== Demand Prediction Agent — Test Metrics ===")
display(metrics_df)

In [ ]:
# Actual vs Predicted utilization (spot-check model quality)
fig, ax = plt.subplots(figsize=(6, 6))
sample = min(2000, len(test))
idx = np.random.choice(len(test), sample, replace=False)
ax.scatter(test[TARGET_UTILIZATION].iloc[idx], pred_util[idx], alpha=0.3, s=10)
ax.plot([0, 1], [0, 1], "r--", label="Perfect prediction")
ax.set_xlabel("Actual Utilization")
ax.set_ylabel("Predicted Utilization")
ax.set_title("Demand Prediction Agent: Utilization")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Importance (interpretability for presentation)

In [ ]:
importance = pd.Series(model_util.feature_importances_, index=feature_cols).sort_values(ascending=True)
importance.tail(12).plot(kind="barh", figsize=(8, 5), title="Top Feature Importances — Utilization Model")
plt.tight_layout()
plt.show()

## 6. Save Models & Predictions (for Tariff & Monitoring Agents)

In [ ]:
joblib.dump(model_util, config.MODELS_DIR / "model_utilization.pkl")
joblib.dump(model_demand, config.MODELS_DIR / "model_demand.pkl")
joblib.dump(model_load, config.MODELS_DIR / "model_load.pkl")
joblib.dump(model_congestion, config.MODELS_DIR / "model_congestion.pkl")

with open(config.MODELS_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f)

# Forecast table for test period (used by pricing agent)
forecasts = test[["datetime", "station_id", TARGET_UTILIZATION, TARGET_LOAD, "volume_kwh"]].copy()
forecasts = forecasts.rename(columns={TARGET_UTILIZATION: "actual_utilization", TARGET_LOAD: "actual_load"})
forecasts["predicted_utilization"] = np.clip(pred_util, 0, 1)
forecasts["predicted_demand"] = np.clip(pred_demand, 0, None)
forecasts["predicted_load"] = np.clip(pred_load, 0, None)
forecasts["congestion_probability"] = pred_congestion_prob

forecasts.to_csv(config.PREDICTIONS_DIR / "demand_forecasts_test.csv", index=False)
metrics_df.to_csv(config.REPORTS_DIR / "demand_prediction_metrics.csv")

with open(config.REPORTS_DIR / "demand_prediction_metrics.json", "w") as f:
    json.dump({k: v for k, v in metrics.items()}, f, indent=2)

print("Saved models and forecasts.")